# RoNIN `a054_1` — first strict public-sequence replay

## tl;dr

- The official 3.2 GB unseen-subject archive was not downloaded in full. HTTP Range transferred about 67.1 MB for one sequence and verified ZIP CRC plus extracted-member SHA-256.
- Estimators received only raw Android IMU-device accelerometer, gyroscope, Game Rotation Vector, and raw IMU timestamps.
- Tango pose, synchronized time, `start_frame`, and `imu_time_offset` were isolated to evaluation.
- B0 was unsupported because no product-compatible step-event stream was admitted.
- All supported B1 ideal runs failed a catastrophic distance, heading, endpoint, or mirror gate at both 50 and 100 Hz. Turn MAE was also high at about 38–44 degrees over 76 events.
- Decision: stop B0/B1 v1.0.0 as product candidates, do not start a personal pilot, and keep Android lifecycle feasibility as a separate unknown.

## Context & Methods

The official custom license is non-commercial scientific-research only and prohibits commercial product use and redistribution without SFU permission. The exact license URL and SHA-256 are in the committed artifact specification.

### Assumptions and claim boundary

1. `raw/imu/acce`, `raw/imu/gyro`, and `raw/imu/game_rv` preserve Android API semantics; the HDF quaternion is explicitly reordered from `w,x,y,z` to Android-like `x,y,z,w`.
2. Source rates above 200 Hz are downsampled causally to 50/100 Hz and are not product requirements.
3. Callback timestamps and Android sensor capability metadata are absent, so this sequence cannot pass lifecycle or device-support gates.
4. Tango orientation and the evaluation-only `align_tango_to_body` label define body-heading truth and the evaluation frame. No ICP or later shape alignment is applied.
5. One public sequence can reject these configurations but cannot prove unseen-device/user generalization.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

repository_root = Path.cwd()
report_path = Path("/outputs/ronin-a054_1-replay.json")
sequence_root = Path("/data/ronin/a054_1")
manifest_path = repository_root / "research" / "pdr" / "datasets" / "manifests" / "ronin-a054_1.json"

if not report_path.exists():
    subprocess.run(
        [
            sys.executable,
            "research/pdr/scripts/replay_ronin_sequence.py",
            "--sequence-root", str(sequence_root),
            "--output", str(report_path),
        ],
        check=True,
    )

report = json.loads(report_path.read_text(encoding="utf-8"))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(
    f"records={report['record_count']} supported={report['supported_record_count']} "
    f"future_violations={report['future_sample_violations']}"
)
print(
    f"range_bytes={manifest['artifact']['http_range_bytes_transferred']} "
    f"archive_bytes={manifest['artifact']['archive_size_bytes']}"
)

records=32 supported=24 future_violations=0
range_bytes=67114615 archive_bytes=3211376453


## Results

### Ideal raw-stream records

In [2]:
print(f"{'rate':>4} {'profile':<16} {'supported':>9} {'truth m':>9} {'estimate m':>10} {'scale':>8} {'drift':>8} {'heading':>9} {'mirror':>7}")
print("-" * 94)
for record in report["records"]:
    if record["scenario"] != "ideal-raw":
        continue
    metrics = record["metrics"]
    if not record["supported"]:
        print(f"{record['target_rate_hz']:>4} {record['capability_profile']:<16} {'no':>9} {'-':>9} {'-':>10} {'-':>8} {'-':>8} {'-':>9} {'-':>7}")
        continue
    print(
        f"{record['target_rate_hz']:>4} {record['capability_profile']:<16} {'yes':>9} "
        f"{metrics['truth_distance_m']:>9.1f} {metrics['estimated_distance_m']:>10.1f} "
        f"{metrics['distance_scale_error']:>8.3f} {metrics['endpoint_drift_ratio']:>8.3f} "
        f"{metrics['heading_mae_deg']:>9.1f} {str(metrics['mirrored']):>7}"
    )

rate profile          supported   truth m estimate m    scale    drift   heading  mirror
----------------------------------------------------------------------------------------------
  50 step-enabled            no         -          -        -        -         -       -
  50 imu6                   yes     381.2      570.5    0.497    0.111      78.4   False
  50 platform-fused         yes     381.2      570.5    0.497    0.991      75.9    True
  50 step-enabled           yes     381.2      570.5    0.497    0.991      75.9    True
 100 step-enabled            no         -          -        -        -         -       -
 100 imu6                   yes     381.2      613.4    0.609    0.122      77.0   False
 100 platform-fused         yes     381.2      613.4    0.609    1.097      76.1    True
 100 step-enabled           yes     381.2      613.4    0.609    1.097      76.1    True


### Acceptance assertions

In [3]:
assert report["record_count"] == 32
assert report["supported_record_count"] == 24
assert report["future_sample_violations"] == 0
assert report["decision"] == "benchmark-only-not-product-go"
assert manifest["license"]["commercial_product_use"] == "prohibited"

b0 = [record for record in report["records"] if record["estimator"].startswith("B0")]
assert len(b0) == 8 and all(not record["supported"] for record in b0)

supported = [record for record in report["records"] if record["supported"]]
for rate in (50, 100):
    for estimator in {record["estimator"] for record in supported if record["target_rate_hz"] == rate}:
        selected = {
            record["scenario"]: record
            for record in supported
            if record["target_rate_hz"] == rate and record["estimator"] == estimator
        }
        assert selected["ideal-raw"]["metrics"] == selected["batch-250ms"]["metrics"]
        assert (
            selected["gap-600ms"]["metrics"]["maximum_uncertainty_m"]
            > selected["ideal-raw"]["metrics"]["maximum_uncertainty_m"]
        )

ideal_b1 = [
    record
    for record in supported
    if record["scenario"] == "ideal-raw" and record["estimator"].startswith("B1")
]
assert len(ideal_b1) == 6 and all(record["failure_flags"] for record in ideal_b1)

manifest_results = {
    (item["target_rate_hz"], item["profile"]): item
    for item in manifest["replay"]["ideal_results"]
}
for record in ideal_b1:
    expected = manifest_results[(record["target_rate_hz"], record["capability_profile"])]
    for manifest_key, metric_key in (
        ("truth_distance_m", "truth_distance_m"),
        ("estimated_distance_m", "estimated_distance_m"),
        ("distance_scale_error", "distance_scale_error"),
        ("endpoint_drift_ratio", "endpoint_drift_ratio"),
        ("heading_mae_deg", "heading_mae_deg"),
        ("turn_mae_deg", "turn_angle_mae_deg"),
        ("evaluated_turn_count", "evaluated_turn_count"),
    ):
        assert expected[manifest_key] == record["metrics"][metric_key]
    assert expected["mirrored"] == record["metrics"]["mirrored"]
print("All license, leakage, capability, batching, gap, and stop-boundary assertions passed.")

All license, leakage, capability, batching, gap, and stop-boundary assertions passed.


## Takeaways

1. The raw Android-field adapter and label-isolation gates work on a real HDF5 sequence at both target rates.
2. The current step detector/stride rule estimates roughly 570–613 m for about 381 m of truth, so low endpoint drift in `imu6` does not imply acceptable distance.
3. Game Rotation Vector produces mirrored, high-drift paths here, reinforcing that device orientation is not body heading.
4. Callback batching independence and explicit gap uncertainty work, but real callback gaps and screen-off behavior are absent from the artifact.
5. Confidence is sufficient to stop these exact baseline configurations. It is not sufficient for a product Go, a personal pilot, or an Android feasibility claim.